# 🚀 Advanced & Emerging Topics — Hands-On Lab## MCP, compound systems, knowledge graphs, distillation, MoE, local LLMs, LLMOps.**Setup:** `pip install openai numpy`**Cost:** ~$0.30 to run everything**Programs:** MCP server pattern, compound AI pipeline, knowledge graph traversal, context compression, distillation pipeline, MoE calculator, local LLM config, LLMOps system.

In [ ]:
# SETUP
from openai import OpenAI
import json, time, hashlib, random, re
from datetime import datetime
from collections import defaultdict

client = OpenAI()

def ask(prompt, model='gpt-4o-mini', temperature=0):
    r = client.chat.completions.create(model=model, temperature=temperature,
        messages=[{'role':'user','content':prompt}])
    return r.choices[0].message.content.strip(), r.usage.total_tokens

print('Setup complete!')

---# 1️⃣ MCP Server Pattern (USB-C for AI Tools)**Analogy:** Before USB-C, every device had a different cable. MCP = one standard protocol for connecting ANY tool to ANY LLM.**What we build:** An MCP-style tool server with tool registry, discovery, and execution.

In [ ]:
# MCP PATTERN: Standard tool interface
# In production: use FastMCP (pip install fastmcp)
# Here: build the pattern from scratch to understand it

class MCPServer:
    def __init__(self, name):
        self.name = name
        self.tools = {}
    
    def register_tool(self, name, description, func, params):
        self.tools[name] = {
            'description': description,
            'function': func,
            'parameters': params,
        }
    
    def discover(self):
        return [{
            'name': name,
            'description': info['description'],
            'parameters': info['parameters'],
        } for name, info in self.tools.items()]
    
    def call(self, tool_name, arguments):
        if tool_name not in self.tools:
            return {'error': f'Tool {tool_name} not found'}
        return self.tools[tool_name]['function'](**arguments)

# Build a knowledge base server
server = MCPServer('Company KB')

server.register_tool(
    'search_docs', 'Search company knowledge base for policies and info.',
    lambda query: {'refund':'30 days standard. Electronics 15 days.','price':'Pro $49/mo.'}.get(
        next((k for k in ['refund','price'] if k in query.lower()), ''), 'No results.'),
    {'query': 'string'}
)

server.register_tool(
    'get_user', 'Look up customer account by ID.',
    lambda user_id: {'123': 'Alice, Pro plan', '456': 'Bob, Enterprise'}.get(user_id, 'Not found'),
    {'user_id': 'string'}
)

print('MCP SERVER PATTERN')
print('=' * 60)

# Discovery: client asks 'what can you do?'
print(f'\n  Server: {server.name}')
print(f'  Tools available:')
for tool in server.discover():
    print(f'    - {tool["name"]}: {tool["description"]}')

# Execution: client calls a tool
print(f'\n  Calling search_docs("refund"):')
print(f'    Result: {server.call("search_docs", {"query": "refund"})}')
print(f'  Calling get_user("123"):')
print(f'    Result: {server.call("get_user", {"user_id": "123"})}')

print(f'\n  In production: pip install fastmcp. 20 lines for a real server.')
print(f'  Any MCP client (Claude, GPT, local) connects to any MCP server.')
print(f'  Build once, connect everywhere.')

---# 2️⃣ Compound AI System (Multiple Specialists Working Together)**Analogy:** A hospital with specialists. Classifier (triage nurse) → retriever (librarian) → generator (doctor) → guardrails (pharmacist checks prescription). Each does ONE thing excellently.**What we build:** A 4-component pipeline: classify → retrieve → generate → safety check. Track cost per component.

In [ ]:
# COMPOUND AI: Multiple specialized components
# Each component = best tool for that job

def component_classifier(query):
    # In production: fine-tuned BERT ($0.0001). Here: rules for demo
    complex_words = ['compare','analyze','design','explain how']
    if any(w in query.lower() for w in complex_words): return 'complex'
    return 'simple'

def component_retriever(query):
    # In production: BM25 + dense hybrid (free/cheap)
    db = {'refund':'30 days. Electronics 15 days.','price':'Pro $49/mo. Enterprise $199/mo.'}
    for k,v in db.items():
        if k in query.lower(): return v
    return 'No relevant docs.'

def component_generator(query, context, complexity):
    model = 'gpt-4o-mini' if complexity == 'simple' else 'gpt-4o-mini'  # Use gpt-4o for complex in prod
    prompt = f'Context: {context}\nQuestion: {query}\nAnswer concisely.'
    answer, tokens = ask(prompt, model=model)
    cost = tokens / 1_000_000 * (0.15 if complexity == 'simple' else 5.0)
    return answer, cost

def component_guardrail(answer):
    banned = ['internal pricing', 'employee names', 'competitor']
    for b in banned:
        if b in answer.lower(): return 'I can only discuss our products.', True
    return answer, False

def compound_pipeline(query):
    # Stage 1: Classify
    complexity = component_classifier(query)
    # Stage 2: Retrieve
    context = component_retriever(query)
    # Stage 3: Generate
    answer, cost = component_generator(query, context, complexity)
    # Stage 4: Safety
    safe_answer, blocked = component_guardrail(answer)
    return {'complexity': complexity, 'answer': safe_answer, 'cost': cost, 'blocked': blocked}

print('COMPOUND AI SYSTEM')
print('=' * 60)

queries = [
    'What is the refund policy?',
    'How much is Pro?',
    'Compare Pro and Enterprise plans in detail',
    'Tell me about competitor products',
]

total_cost = 0
for q in queries:
    r = compound_pipeline(q)
    total_cost += r['cost']
    blocked = ' [BLOCKED]' if r['blocked'] else ''
    print(f'  [{r["complexity"]:7s}] ${r["cost"]:.5f} | {q[:40]}...')
    print(f'           → {r["answer"][:50]}...{blocked}')

print(f'\n  Total cost: ${total_cost:.5f}')
print(f'  All-GPT-4o: ${len(queries) * 0.015:.5f} (if no routing)')
print(f'\n  Each component is independently testable and replaceable.')

---# 3️⃣ Knowledge Graph (Follow Chains Across Documents)**Analogy:** Regular RAG finds individual pages. But 'Who does Bob report to?' requires following a chain: Bob → Alice → Carol (CEO). That chain spans 3 documents.**What we build:** A simple knowledge graph with entity extraction and multi-hop traversal.

In [ ]:
# KNOWLEDGE GRAPH: Entities + relationships + traversal

class KnowledgeGraph:
    def __init__(self):
        self.entities = {}
        self.relationships = []
    
    def add_entity(self, name, etype):
        self.entities[name] = {'type': etype}
    
    def add_relationship(self, source, relation, target):
        self.relationships.append({'source':source, 'relation':relation, 'target':target})
    
    def traverse(self, start, relation, max_hops=5):
        path = [start]
        current = start
        for _ in range(max_hops):
            found = [r['target'] for r in self.relationships if r['source']==current and r['relation']==relation]
            if not found: break
            current = found[0]
            path.append(current)
        return path
    
    def find_connections(self, entity):
        outgoing = [(r['relation'], r['target']) for r in self.relationships if r['source']==entity]
        incoming = [(r['source'], r['relation']) for r in self.relationships if r['target']==entity]
        return outgoing, incoming

# Build from 'documents'
print('KNOWLEDGE GRAPH')
print('=' * 60)

kg = KnowledgeGraph()

# Entities (extracted from documents by LLM in production)
for name, etype in [('Bob','person'),('Alice','person'),('Carol','person'),
                     ('Acme Corp','company'),('Project Alpha','project'),('Engineering','department')]:
    kg.add_entity(name, etype)

# Relationships
for s, r, t in [('Bob','reports_to','Alice'),('Alice','reports_to','Carol'),
                ('Carol','ceo_of','Acme Corp'),('Bob','works_on','Project Alpha'),
                ('Alice','leads','Engineering'),('Project Alpha','belongs_to','Engineering')]:
    kg.add_relationship(s, r, t)

print('\n  Graph:')
for r in kg.relationships:
    print(f'    {r["source"]} --{r["relation"]}--> {r["target"]}')

# Multi-hop queries
print(f'\n  MULTI-HOP QUERIES:')
chain = kg.traverse('Bob', 'reports_to')
print(f'    Q: Who does Bob ultimately report to?')
print(f'    Chain: {" → ".join(chain)}')
print(f'    Answer: {chain[-1]} ({len(chain)-1} hops)')

out, inc = kg.find_connections('Alice')
print(f'\n    Q: What is Alice connected to?')
print(f'    Outgoing: {out}')
print(f'    Incoming: {inc}')

print(f'\n  Regular RAG: finds Bob→Alice but misses Alice→Carol.')
print(f'  GraphRAG: follows the chain across documents.')
print(f'  Use when 20%+ of queries need cross-document connections.')

---# 4️⃣ Distillation + LLMOps (Train Small from Big + Version Everything)**Distillation:** Master chef (GPT-4) cooks 10K meals. Student chef watches and learns. Student makes 90% as good meals at 1/50th the price.**LLMOps:** DevOps for AI. Version prompts → evaluate → deploy → monitor. Every change tracked.

In [ ]:
# DISTILLATION: Generate training data from a strong model
# LLMOPS: Version, evaluate, deploy, monitor

# ---------- DISTILLATION PIPELINE ----------
print('DISTILLATION PIPELINE')
print('=' * 60)

def distillation_pipeline(task_prompts, teacher='gpt-4o-mini'):
    training_data = []
    for prompt in task_prompts:
        teacher_answer, tokens = ask(prompt, model=teacher)
        training_data.append({
            'messages': [
                {'role':'system','content':'You are a helpful classifier.'},
                {'role':'user','content':prompt},
                {'role':'assistant','content':teacher_answer},
            ]
        })
    return training_data

sample_tasks = [
    'Classify sentiment: Amazing product!',
    'Classify sentiment: Terrible quality, broke immediately.',
    'Classify sentiment: It works, nothing special.',
    'Classify sentiment: Best purchase of the year!',
    'Classify sentiment: Would not recommend to anyone.',
]

print(f'\n  Teacher model generates {len(sample_tasks)} training examples...')
data = distillation_pipeline(sample_tasks)
for d in data:
    user = d['messages'][1]['content'][:35]
    asst = d['messages'][2]['content'][:20]
    print(f'    {user:35s} → {asst}')

print(f'\n  Next: fine-tune a small model (Llama-8B) on these {len(data)} examples.')
print(f'  At scale: 10K examples, cost ~$5 to generate, ~$50 to fine-tune.')
print(f'  Result: 90% of teacher quality at 1/50th inference cost.')

# ---------- LLMOPS ----------
print(f'\n\nLLMOPS (Version + Evaluate + Deploy)')
print('=' * 60)

class LLMOps:
    def __init__(self):
        self.versions = {}
        self.audit = []
    
    def save(self, name, text, author, metrics=None):
        if name not in self.versions: self.versions[name] = []
        v = len(self.versions[name]) + 1
        h = hashlib.md5(text.encode()).hexdigest()[:8]
        self.versions[name].append({'v':v,'text':text,'hash':h,'author':author,'metrics':metrics or {}})
        self.audit.append(f'{name} v{v} [{h}] by {author}')
        return v
    
    def evaluate(self, name, v, tests, llm_func):
        prompt = self.versions[name][v-1]['text']
        passed = sum(1 for t in tests if t['expected'].lower() in llm_func(prompt.replace('{text}',t['input'])).lower())
        acc = passed / len(tests)
        self.versions[name][v-1]['metrics']['accuracy'] = acc
        return acc
    
    def deploy_decision(self, name, v, min_accuracy=0.80):
        acc = self.versions[name][v-1]['metrics'].get('accuracy', 0)
        if acc >= min_accuracy:
            self.audit.append(f'DEPLOYED {name} v{v} (accuracy {acc:.0%})')
            return True
        self.audit.append(f'BLOCKED {name} v{v} (accuracy {acc:.0%} < {min_accuracy:.0%})')
        return False

ops = LLMOps()
ops.save('sentiment', 'Classify POS/NEG: {text}', 'alice')
ops.save('sentiment', 'Classify POS/NEG/NEUTRAL: {text}', 'bob')

tests = [{'input':'Great!','expected':'POS'},{'input':'Terrible!','expected':'NEG'},{'input':'Ok.','expected':'NEU'}]

def mock_llm(p): return 'POSITIVE' if 'great' in p.lower() else 'NEGATIVE' if 'terrible' in p.lower() else 'NEUTRAL'

for v in [1, 2]:
    acc = ops.evaluate('sentiment', v, tests, mock_llm)
    deployed = ops.deploy_decision('sentiment', v)
    status = '✅ DEPLOYED' if deployed else '🚫 BLOCKED'
    print(f'  v{v}: accuracy={acc:.0%} → {status}')

print(f'\n  Audit trail: {ops.audit}')
print(f'\n  Cycle: version → evaluate → deploy/block → monitor → iterate')

---# 5️⃣ MoE Calculator + Local LLM Guide**MoE:** 8 expert lawyers but only 2 needed per case. 47B knowledge at 13B speed. Best self-hosting value.**Local LLMs:** Run on your laptop. No API. No cost. No data leaving your machine.

In [ ]:
# MOE CALCULATOR + LOCAL LLM GUIDE

print('MIXTURE OF EXPERTS (MoE)')
print('=' * 60)

moe_models = [
    ('Mixtral 8x7B', 47, 13, 8, 2),
    ('Mixtral 8x22B', 141, 39, 8, 2),
    ('DeepSeek-V2', 236, 21, 160, 6),
]

print(f'\n  {"Model":20s} {"Total":>8s} {"Active":>8s} {"Experts":>10s} {"Speedup":>8s}')
print('  ' + '-' * 58)
for name, total, active, experts, top_k in moe_models:
    speedup = total / active
    print(f'  {name:20s} {total:>6}B {active:>6}B {top_k}/{experts:>3} active {speedup:>6.1f}x')

print(f'\n  Key insight: {moe_models[0][0]} has {moe_models[0][1]}B knowledge')
print(f'  but only {moe_models[0][2]}B compute per token = big brain, small bill.')

# GPU REQUIREMENTS
print(f'\n\n  GPU REQUIREMENTS (Mixtral 8x7B):')
for quant, size, gpus in [('FP16',94,2),('INT8',47,1),('INT4',24,1)]:
    print(f'    {quant:6s}: {size:3d}GB → {gpus} GPU(s)')

# LOCAL LLMS
print(f'\n\nLOCAL LLMs')
print('=' * 60)

models = [
    ('Phi-3-mini (3.8B)', '4GB', 'Basic tasks, fast'),
    ('Llama-3-8B', '8GB', 'Good general purpose'),
    ('Llama-3-13B', '16GB', 'Better quality'),
    ('Mixtral 8x7B', '32GB', 'Near GPT-3.5 quality'),
    ('Llama-3-70B (Q4)', '40GB', 'Best local quality'),
]

print(f'\n  {"Model":25s} {"RAM":>6s}   {"Notes"}')
print('  ' + '-' * 55)
for name, ram, note in models:
    print(f'  {name:25s} {ram:>6s}   {note}')

print(f'\n  Quick start: ollama run llama3 (one command!)')
print(f'  API: ollama serve → OpenAI-compatible on localhost:11434')
print(f'  Same client code works: just change base_url.')

print(f'\n  COST (1M queries/month):')
for name, cost in [('GPT-4o API','$15,000'),('GPT-4o-mini API','$500'),
                    ('Local Llama-70B','$300 (GPU)'),('Local Llama-8B','$0 (laptop)')]:
    print(f'    {name:20s} {cost}')

---# 🏁 Summary — Advanced Topics Cheat Sheet| Topic | Key Takeaway ||-------|-------------|| **MCP** | USB-C for AI tools. Build once, connect everywhere. || **Compound AI** | Multiple specialists > one generalist. 80-90% cheaper. || **Knowledge Graphs** | Follow chains across documents. Use when 20%+ queries need it. || **Long Context** | Try RAG first. Then recursive summarization. Then compression. || **Distillation** | GPT-4 teaches small model. 90% quality at 1/50th cost. || **MoE** | 47B knowledge at 13B speed. Best self-hosting value. || **Local LLMs** | Free, private, always on. Good for dev + privacy-sensitive tasks. || **LLMOps** | Version → evaluate → deploy → monitor. Treat prompts as code. |